# MOFA+ evaluation

Scope: three analyses, all scored with the same `calculate_metrics()` /
`pooled_correlations()` functions and the same `build_masked_scenario()`
masking used for the 25-method benchmark (`benchmark_imputation_clinical.ipynb` /
`evaluate_imputation_methods_multiomic_clinical.ipynb`), so that MOFA+'s results land as
extra rows in those same tables rather than sitting off in a separate, hard-to-compare
analysis.

1. **Per-matrix baseline ("MOFA+_native")**:
   MOFA+ is trained as a single-view model on one matrix at a
   time (`dlbcl_rna`, `dlbcl_protein`, `crc_protein`), using its own native
   missing-value handling on the masked matrix, and scored at exactly the
   withheld positions.

2. **MNAR-aware pre-impute ablation**: on the two protein matrices only,
   where the MNAR/detection-limit pattern is real. We pre-impute the masked
   matrix with QRILC and, separately, with MinProb (both already part of the
   25-method roster), and *then* train MOFA+ on the now-complete matrix. This
   tests whether an MNAR-aware pre-imputer beats MOFA+'s own
   MAR-assuming native handling on exactly the missingness pattern where
   that assumption is most likely to break down. 

3. **Joint two-view run ("MOFA+_joint")**: DLBCL RNA and DLBCL protein
   trained *together* as MOFA+'s two views, exercising its actual
   multi-omics value proposition rather than the per-matrix baseline. Each
   view's reconstruction is scored against that view's own truth.
   `crc_protein` has no joint partner (a different patient cohort, with no
   shared samples), so it only ever gets a "MOFA+_native" row.

In [ ]:
# --- Cell 1: cap CPU thread usage, imports, path resolution, and a guard for the mofapy2 environment ---
import os
import time
import warnings
from pathlib import Path


def _cluster_cpu_budget(default=4):
    """How many CPU threads/workers this job should actually use.

    Priority: an explicit IMPUTATION_CPU_BUDGET override (for interactive use
    on an unmanaged, possibly-shared node - cpu34/cpu35, or the gpu25-27
    prototyping nodes - where Slurm sets none of the variables below) beats
    the Slurm allocation (when this is an actual sbatch job on gpu-unlimited),
    which beats this process's own CPU affinity, which beats a conservative
    default. Also returns where the number came from, for the printed message
    below."""
    override = os.environ.get("IMPUTATION_CPU_BUDGET")
    if override:
        try:
            return max(1, int(override)), "IMPUTATION_CPU_BUDGET override"
        except ValueError:
            print(f"IMPUTATION_CPU_BUDGET={override!r} isn't a valid integer - ignoring it.")

    for var in ("SLURM_CPUS_PER_TASK", "SLURM_CPUS_ON_NODE"):
        val = os.environ.get(var)
        if val:
            try:
                return max(1, int(val)), var
            except ValueError:
                pass
    val = os.environ.get("SLURM_JOB_CPUS_PER_NODE")
    if val:
        import re
        m = re.match(r"(\d+)", val)
        if m:
            return max(1, int(m.group(1))), "SLURM_JOB_CPUS_PER_NODE"

    try:
        return max(1, len(os.sched_getaffinity(0))), "CPU affinity (no Slurm allocation, no override detected)"
    except AttributeError:
        return max(1, os.cpu_count() or default), "os.cpu_count() (no Slurm allocation, no override detected)"


# Must run before numpy/pandas/mudata/mofapy2 are imported below - MOFA+'s
# training is numpy-driven under the hood (gpu_mode=False, see Cell 7), so it
# reads these same thread env vars too.
N_JOBS, _cpu_budget_source = _cluster_cpu_budget()
for _var in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS",
             "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[_var] = str(N_JOBS)
print(f"CPU budget for this session: {N_JOBS} threads (source: {_cpu_budget_source})")
if _cpu_budget_source.startswith(("CPU affinity", "os.cpu_count")):
    print(
        "  no Slurm allocation detected - this looks like an unmanaged node "
        "(cpu34/cpu35, or a gpu25-27 prototyping session). If sharing it "
        "with others, set IMPUTATION_CPU_BUDGET to a smaller number "
        "before starting this kernel, e.g. `export IMPUTATION_CPU_BUDGET=16`."
    )

import numpy as np
import pandas as pd
import mudata as mu
from scipy.stats import norm, truncnorm, spearmanr

mu.set_options(pull_on_update=False)
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

RNG_SEED = 0  # keep in sync with benchmark_imputation_clinical.ipynb/py / 
              # evaluate_imputation_methods_multiomic_clinical.ipynb

RUN_MOFAPLUS = True
MOFAPLUS_AVAILABLE = False
if RUN_MOFAPLUS:
    try:
        from mofapy2.run.entry_point import entry_point  # noqa: F401
        MOFAPLUS_AVAILABLE = True
        print("mofapy2 available: MOFA+ enabled.")
    except Exception as exc:
        print(
            f"MOFA+ disabled ({type(exc).__name__}: {exc}). Requires `pip install mofapy2` "
            "in this environment. The rest of this notebook will still run (matrix rebuild, "
            "masking, pre-imputers) but every MOFA+ training cell will skip with a clear message."
        )

try:
    THIS_DIR = Path(__file__).resolve().parent
except NameError:
    THIS_DIR = Path.cwd()

BASE_DIR = THIS_DIR.parent  # the project root, e.g. /home/rnjue/Imputationv2

CANDIDATE_PROCESSED_DIRS = []
if os.environ.get("IMPUTATION_OUTPUT_DIR"):
    CANDIDATE_PROCESSED_DIRS.append(Path(os.environ["IMPUTATION_OUTPUT_DIR"]).expanduser())
CANDIDATE_PROCESSED_DIRS += [
    BASE_DIR / "data" / "processed",
    Path.home() / "reuben_imputation" / "processed",
    Path("/projects/datasets_BIO/reuben_imputation/processed"),
    THIS_DIR / "processed",
]
PROCESSED_DIR = next((d for d in CANDIDATE_PROCESSED_DIRS if d.exists()), None)
if PROCESSED_DIR is None:
    raise FileNotFoundError(
        "no 'processed' directory found among: " + ", ".join(str(d) for d in CANDIDATE_PROCESSED_DIRS) +
        ". Run read_multiomic_Data_clinical.ipynb through Cell 14 first, or set IMPUTATION_OUTPUT_DIR."
    )

CANDIDATE_RESULTS_DIRS = [
    BASE_DIR / "results",
    PROCESSED_DIR.parent / "imputation_results",
    Path.home() / "reuben_imputation" / "imputation_results",
]
RESULTS_DIR = next((d for d in CANDIDATE_RESULTS_DIRS if d.exists()), None)
if RESULTS_DIR is None:
    raise FileNotFoundError(
        "no imputation_results directory found among: " + ", ".join(str(d) for d in CANDIDATE_RESULTS_DIRS) +
        ". Run benchmark_imputation_clinical.ipynb first."
    )
# Deliberately the same files that benchmark_imputation_clinical.ipynb's Cell 15 and
# evaluate_imputation_methods_clinical.ipynb already read and write. MOFA+'s rows
# get appended alongside the 25 methods' rows, so it shows up as just one more Method in
# the exact same tables, rather than a separate analysis you'd have to cross-reference.
PILOT_CHECKPOINT_PATH = RESULTS_DIR / "pilot_checkpoint.csv"
PILOT_MATRICES_DIR = RESULTS_DIR / "pilot_imputed_matrices"
PILOT_MATRICES_DIR.mkdir(exist_ok=True)
print("processed dir:", PROCESSED_DIR, "| results dir:", RESULTS_DIR)

mdata = mu.read_h5mu(PROCESSED_DIR / "dlbcl_mudata_filtered.h5mu")
crc_df = pd.read_csv(PROCESSED_DIR / "crc_proteomics_samples_x_features.csv", index_col=0)


def to_dense(X):
    return X.toarray() if hasattr(X, "toarray") else np.asarray(X)

CPU budget for this session: 80 threads (source: CPU affinity (no Slurm allocation, no override detected))
  no Slurm allocation detected - this looks like an unmanaged node (cpu34/cpu35, or a gpu25-27 prototyping session). If you're sharing it with others right now, set IMPUTATION_CPU_BUDGET to a smaller number before starting this kernel, e.g. `export IMPUTATION_CPU_BUDGET=16`.
mofapy2 available: MOFA+ enabled.
processed dir: /home/rnjue/Imputationv2/data/processed | results dir: /home/rnjue/Imputationv2/results


In [2]:
# --- Cell 2: rebuild the three per-matrix truth matrices ------
# features x samples, restricted to the top 500 most-variable features.
# Feeds the per-matrix baseline (analysis 1) and the
# pre-impute ablation (analysis 2).
rna_adata = mdata.mod["1_rna"]
assert (rna_adata.obs_names == mdata.obs_names).all()
profiled_mask = mdata.obs["rna_profiled"].to_numpy()

rna_full = pd.DataFrame(
    to_dense(rna_adata.X)[profiled_mask].T, index=rna_adata.var_names, columns=rna_adata.obs_names[profiled_mask]
)
prot_full = pd.DataFrame(to_dense(mdata.mod["2_prot"].X).T, index=mdata.mod["2_prot"].var_names, columns=mdata.obs_names)
crc_full = crc_df.T

N_TOP_FEATURES = 500  # matches benchmark_imputation_clinical.ipynb/py / 
                      # evaluate_imputation_methods_clinical.ipynb


def top_variable_features(df, n):
    if df.shape[0] <= n:
        return df
    top_idx = df.var(axis=1, skipna=True).sort_values(ascending=False).index[:n]
    return df.loc[df.index.isin(top_idx)]


MATRICES = {
    "dlbcl_rna": top_variable_features(rna_full, N_TOP_FEATURES),
    "dlbcl_protein": top_variable_features(prot_full, N_TOP_FEATURES),
    "crc_protein": top_variable_features(crc_full, N_TOP_FEATURES),
}
PROTEIN_MATRICES = {"dlbcl_protein", "crc_protein"}
for name, df in MATRICES.items():
    print(f"{name}: {df.shape}")

dlbcl_rna: (500, 318)
dlbcl_protein: (500, 332)
crc_protein: (500, 997)


In [3]:
# --- Cell 3: rebuild the full 332-sample DLBCL RNA matrix (analysis 3 only) ---
# MATRICES["dlbcl_rna"] above is restricted to the 318 RNA-profiled samples, matching
# how every conventional method was benchmarked. For the joint two-view run that
# restriction is wrong, though: MOFA is explicitly designed to handle samples that are
# profiled in only SOME views, and `mdata.mod["1_rna"].X` already carries all 332 samples, with the
# 14 unprofiled ones presumably NaN (the same union indexing that `rna_adata.obs_names
# == mdata.obs_names` already asserted in Cell 2). So this is the same underlying data,
# just without dropping those 14 rows. `dlbcl_protein` already has all 332 samples, so
# it needs no filtering.
rna_full_332 = pd.DataFrame(to_dense(rna_adata.X).T, index=rna_adata.var_names, columns=rna_adata.obs_names)
JOINT_MATRICES = {
    "dlbcl_rna": top_variable_features(rna_full_332, N_TOP_FEATURES),
    "dlbcl_protein": MATRICES["dlbcl_protein"],  # already all 332 samples
}
for name, df in JOINT_MATRICES.items():
    n_all_nan_cols = int((df.isna().all(axis=0)).sum())
    print(f"{name} (joint, unfiltered): {df.shape} | {n_all_nan_cols} fully-unprofiled sample columns")
assert list(JOINT_MATRICES["dlbcl_rna"].columns) == list(JOINT_MATRICES["dlbcl_protein"].columns), (
    "the two joint views must share identical sample ordering -- mdata's union indexing should guarantee this"
)

dlbcl_rna (joint, unfiltered): (500, 332) | 14 fully-unprofiled sample columns
dlbcl_protein (joint, unfiltered): (500, 332) | 0 fully-unprofiled sample columns


In [4]:
# --- Cell 4: masking functions and PILOT_ROW ------------------
def generate_mnar_weighted_positions(X, n_to_mask, slope=-1.5, rng=None, exclude_mask=None):
    if rng is None:
        rng = np.random.default_rng()
    eligible = X.notna().values
    if exclude_mask is not None:
        eligible = eligible & ~exclude_mask.values
    positions = np.argwhere(eligible)
    if n_to_mask == 0:
        return np.empty((0, 2), dtype=int)
    values = X.values[eligible]
    mean_v, std_v = values.mean(), values.std()
    z = np.zeros_like(values) if std_v == 0 else (values - mean_v) / std_v
    weights = np.exp(slope * z)
    weights = weights / weights.sum()
    selected = rng.choice(len(positions), size=n_to_mask, replace=False, p=weights)
    return positions[selected]


def generate_mixed_mask(X, masking_fraction=0.20, mnar_fraction=0.50, slope=-1.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    eligible = X.notna().values
    n_eligible = eligible.sum()
    total_to_mask = int(round(masking_fraction * n_eligible))
    n_mnar = int(round(total_to_mask * mnar_fraction))
    n_mcar = total_to_mask - n_mnar

    mnar_positions = generate_mnar_weighted_positions(X, n_mnar, slope=slope, rng=rng)
    mnar_excl = np.zeros(X.shape, dtype=bool)
    if len(mnar_positions) > 0:
        mnar_excl[mnar_positions[:, 0], mnar_positions[:, 1]] = True
    mnar_excl_df = pd.DataFrame(mnar_excl, index=X.index, columns=X.columns)

    mcar_positions = np.empty((0, 2), dtype=int)
    if n_mcar > 0:
        available = np.argwhere(eligible & ~mnar_excl_df.values)
        n_mcar_clamped = min(n_mcar, len(available))
        mcar_positions = available[rng.choice(len(available), size=n_mcar_clamped, replace=False)]

    mask = np.zeros(X.shape, dtype=bool)
    mechanism = np.full(X.shape, "", dtype=object)
    if len(mnar_positions) > 0:
        mask[mnar_positions[:, 0], mnar_positions[:, 1]] = True
        mechanism[mnar_positions[:, 0], mnar_positions[:, 1]] = "MNAR"
    if len(mcar_positions) > 0:
        mask[mcar_positions[:, 0], mcar_positions[:, 1]] = True
        mechanism[mcar_positions[:, 0], mcar_positions[:, 1]] = "MCAR"
    return (
        pd.DataFrame(mask, index=X.index, columns=X.columns),
        pd.DataFrame(mechanism, index=X.index, columns=X.columns),
    )


def build_masked_scenario(row, X_truth, slope=-1.5):
    rng = np.random.default_rng(int(row["Seed"]))
    mask, mechanism = generate_mixed_mask(
        X_truth, masking_fraction=float(row["Masking_Fraction"]), mnar_fraction=float(row["MNAR_Fraction"]),
        slope=slope, rng=rng,
    )
    return X_truth.mask(mask), mask, mechanism


PILOT_ROW = pd.Series({"Masking_Fraction": 0.10, "MNAR_Fraction": 0.0, "Replicate": 1, "Seed": RNG_SEED})

In [5]:
# --- Cell 5: metrics ---- calculate_metrics and pooled_correlations ------
def calculate_metrics(truth, prediction, mask):
    y_true_mat, y_pred_mat, mask_mat = truth.values, prediction.values, mask.values
    y_true, y_pred = y_true_mat[mask_mat], y_pred_mat[mask_mat]
    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true, y_pred = y_true[valid], y_pred[valid]

    errors = y_true - y_pred
    rmse = np.sqrt(np.mean(errors ** 2)) if len(errors) else np.nan
    mae = np.mean(np.abs(errors)) if len(errors) else np.nan
    pearson_r = np.corrcoef(y_true, y_pred)[0, 1] if len(y_true) > 1 else np.nan

    nrmse_vals = []
    min_masked_per_feature = 5
    for i in range(y_true_mat.shape[0]):
        row_mask = mask_mat[i]
        if row_mask.sum() < min_masked_per_feature:
            continue
        yt, yp = y_true_mat[i, row_mask], y_pred_mat[i, row_mask]
        v = np.isfinite(yt) & np.isfinite(yp)
        yt, yp = yt[v], yp[v]
        if len(yt) < min_masked_per_feature:
            continue
        feat_range = np.nanmax(y_true_mat[i]) - np.nanmin(y_true_mat[i])
        if not np.isfinite(feat_range) or feat_range <= 0:
            continue
        row_rmse = np.sqrt(np.mean((yt - yp) ** 2))
        nrmse_vals.append(row_rmse / feat_range)

    return {
        "RMSE": rmse, "MAE": mae, "Pearson_r": pearson_r,
        "NRMSE": float(np.mean(nrmse_vals)) if nrmse_vals else np.nan,
        "N_Evaluated": len(y_true),
    }


def pooled_correlations(X_truth, X_hat, mask, min_masked_per_feature=5):
    yt_mat, yp_mat, mask_mat = X_truth.values, X_hat.values, mask.values
    raw_true, raw_pred, cen_true, cen_pred = [], [], [], []
    n_features_used = 0
    for i in range(yt_mat.shape[0]):
        row_mask = mask_mat[i]
        if row_mask.sum() < min_masked_per_feature:
            continue
        yt, yp = yt_mat[i, row_mask], yp_mat[i, row_mask]
        v = np.isfinite(yt) & np.isfinite(yp)
        yt, yp = yt[v], yp[v]
        if len(yt) < min_masked_per_feature:
            continue
        raw_true.append(yt)
        raw_pred.append(yp)
        cen_true.append(yt - yt.mean())
        cen_pred.append(yp - yp.mean())
        n_features_used += 1

    if n_features_used == 0:
        return {"Pearson_r_raw_pooled": np.nan, "Spearman_r_raw_pooled": np.nan,
                "Pearson_r_feature_centered_pooled": np.nan, "Spearman_r_feature_centered_pooled": np.nan,
                "N_Features_Pooled": 0}

    raw_true, raw_pred = np.concatenate(raw_true), np.concatenate(raw_pred)
    cen_true, cen_pred = np.concatenate(cen_true), np.concatenate(cen_pred)

    def _pearson(a, b):
        return np.corrcoef(a, b)[0, 1] if len(a) > 1 and np.std(a) > 0 and np.std(b) > 0 else np.nan

    def _spearman(a, b):
        return spearmanr(a, b).correlation if len(a) > 1 and np.std(a) > 0 and np.std(b) > 0 else np.nan

    return {
        "Pearson_r_raw_pooled": _pearson(raw_true, raw_pred),
        "Spearman_r_raw_pooled": _spearman(raw_true, raw_pred),
        "Pearson_r_feature_centered_pooled": _pearson(cen_true, cen_pred),
        "Spearman_r_feature_centered_pooled": _spearman(cen_true, cen_pred),
        "N_Features_Pooled": n_features_used,
    }


def score_full(matrix_name, method_name, X_truth, X_hat, mask, t0):
    """Bundle calculate_metrics and pooled_correlations into a single row, in
    exactly the shape pilot_checkpoint.csv / pilot_evaluation_summary.csv expect."""
    X_hat = X_hat.reindex(index=X_truth.index, columns=X_truth.columns)
    m = calculate_metrics(X_truth, X_hat, mask)
    m.update(pooled_correlations(X_truth, X_hat, mask))
    m.update({"Matrix": matrix_name, "Method": method_name, "Seconds": time.time() - t0,
              **{c: PILOT_ROW[c] for c in ["Masking_Fraction", "MNAR_Fraction", "Replicate", "Seed"]}})
    return m, X_hat

In [6]:
# --- Cell 6: MNAR-aware pre-imputers for the pre-impute-then-MOFA+ ablation ---
# (analysis 2). Specifically QRILC and MinProb, out of the 6 MNAR/left-censored
# methods already in the benchmark: both are probabilistic left-censored imputers
# (Lazar et al. 2016), and the literature's most commonly recommended pre-imputers
# for detection-limit proteomics data - as opposed to the deterministic
# MinDet/LOD/Zero/HalfMin alternatives.
def minprob_impute(Y, q=0.01, sigma_scale=0.3, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    result = Y.copy()
    for col in result.columns:
        observed = result[col].dropna()
        if observed.empty:
            continue
        missing_mask = result[col].isna()
        n_missing = int(missing_mask.sum())
        if n_missing == 0:
            continue
        mu = observed.quantile(q)
        sigma = max(observed.std() * sigma_scale, 1e-6)
        result.loc[missing_mask, col] = rng.normal(loc=mu, scale=sigma, size=n_missing)
    return result


def qrilc_impute(Y, q_tail=0.30, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    result = Y.copy()
    global_sd = np.nanstd(Y.values)
    for feat in result.index:
        observed = result.loc[feat].dropna()
        missing_mask = result.loc[feat].isna()
        n_missing = int(missing_mask.sum())
        if n_missing == 0 or observed.empty:
            continue
        sorted_obs = np.sort(observed.values)
        n_obs = len(sorted_obs)
        n_tail = max(3, int(np.ceil(q_tail * n_obs)))
        tail = sorted_obs[:n_tail]
        pp = (np.arange(1, n_tail + 1) - 0.5) / n_obs
        z = norm.ppf(pp)
        if n_tail >= 3 and np.std(z) > 0:
            slope, intercept = np.polyfit(z, tail, 1)
            sigma_hat, mu_hat = max(slope, 1e-3), intercept
        else:
            mu_hat, sigma_hat = tail.mean(), (global_sd if global_sd > 0 else 1e-3)
        a, b = -np.inf, (sorted_obs[0] - mu_hat) / sigma_hat
        result.loc[feat, missing_mask] = truncnorm.rvs(a, b, loc=mu_hat, scale=sigma_hat, size=n_missing, random_state=rng)
    return result

In [7]:
# --- Cell 7: run_mofaplus()---- trains MOFA+ and reconstructs each view -------
N_FACTORS = 15  # a generous starting point; the ARD and spike-slab priors shrink
                 # unused factors toward zero automatically (adopted from paper's own
                 # rationale for not needing to hand-pick K). 
MOFAPLUS_CONVERGENCE_MODE = "fast"
MOFAPLUS_MAX_ITER = 1000


def run_mofaplus(views_data, group_name="all_samples", n_factors=N_FACTORS, seed=RNG_SEED,
                  convergence_mode=MOFAPLUS_CONVERGENCE_MODE, max_iter=MOFAPLUS_MAX_ITER, verbose=False):
    """Train MOFA+ as a single group with one or more views sharing the same
    sample set, via mofapy2's entry_point, and return each view's reconstruction
    Z_hat @ W_hat.T as a features x samples DataFrame - for this analysis 
    we set mofapy2 to expect samples x features. 

    views_data: dict of {view_name: DataFrame}, each features x samples, all
    sharing identical column (sample) ordering. NaN means missing and is handled
    natively - no other sentinel value, nothing extra to configure.

    Deliberately skips entry_point.impute()'s convenience wrapper, which
    overwrites its own reconstruction with the raw input at every position it
    considers "not missing." That's the right behavior when reading out a
    natively-trained model's imputations, but wrong for the pre-impute ablation
    below, where nothing is left "missing" for the wrapper to skip over.
    Computing Z_hat @ W_hat.T directly (the same math impute() uses internally)
    and letting the CALLER decide which positions to score keeps the native and
    pre-impute pipelines directly comparable.
    """
    from mofapy2.run.entry_point import entry_point

    view_names = list(views_data.keys())
    sample_names = list(views_data[view_names[0]].columns)
    for vn in view_names:
        assert list(views_data[vn].columns) == sample_names, (
            f"view '{vn}' has a different sample ordering than view '{view_names[0]}' - "
            "all views in one MOFA+ run must share identical columns"
        )

    data = [[views_data[vn].T.values] for vn in view_names]  # data[view][group=0] has shape (N_samples, D_features)
    likelihoods = ["gaussian"] * len(view_names)
    features_names = [list(views_data[vn].index) for vn in view_names]  # nested by VIEW
    samples_names = [sample_names]  # nested by GROUP (just one group here)

    ent = entry_point()
    ent.set_data_options(scale_views=False, scale_groups=False, center_groups=True)
    ent.set_data_matrix(
        data, likelihoods=likelihoods, views_names=view_names, groups_names=[group_name],
        samples_names=samples_names, features_names=features_names,
    )
    ent.set_model_options(
        factors=n_factors, spikeslab_weights=True, spikeslab_factors=False,
        ard_weights=True, ard_factors=False,  # ard_factors only matters across multiple GROUPs; we always use 1
    )
    ent.set_train_options(
        iter=max_iter, convergence_mode=convergence_mode, seed=seed,
        gpu_mode=False, verbose=verbose, quiet=not verbose,
    )
    ent.build()
    ent.run()

    W = [w["E"] for w in ent.model.nodes["W"].getExpectations()]
    Z = ent.model.nodes["Z"].getExpectations()["E"]
    reconstructions = {}
    for i, vn in enumerate(view_names):
        pred = Z.dot(W[i].T)  # shape (N_samples, D_features), on the CENTERED scale -
        # set_data_options above was called with center_groups=True, so mofapy2 subtracts
        # each feature's own mean before fitting, and Z @ W.T reconstructs that centered
        # signal rather than the feature's original level. entry_point.impute() normally
        # adds this same per-feature mean back before returning (mirroring MOFA2::impute()'s
        # add_intercept=TRUE), but this function deliberately bypasses impute() for a
        # different reason (see the docstring above) - which means the intercept restoration
        # has to happen explicitly here, or it never happens at all.
        feature_means = views_data[vn].mean(axis=1, skipna=True).values  # the same per-feature mean
        # mofapy2 itself centered by: views_data[vn] is features x samples, so this averages
        # across samples, ignoring the positions that were already missing.
        pred = pred + feature_means[np.newaxis, :]
        reconstructions[vn] = pd.DataFrame(pred.T, index=views_data[vn].index, columns=sample_names)
    return reconstructions, ent

In [ ]:
# --- One-off cleanup: this was done to clear stale MOFA+ results from run_mofaplus() -----
# Cells 8-11's skip/resume check only looks at whether a row and matrix file already
# exist - it has no way to tell that the underlying code changed. Without this cleanup,
# re-running those cells would just read the old, pre-fix results back out of the checkpoints.
MOFAPLUS_METHOD_NAMES = {"MOFA+_native", "MOFA+_after_QRILC", "MOFA+_after_MinProb", "MOFA+_joint"}

# 1) pilot_checkpoint.csv: drop the MOFA+ rows, keep the 25 conventional methods
if PILOT_CHECKPOINT_PATH.exists():
    _ckpt = pd.read_csv(PILOT_CHECKPOINT_PATH)
    _before = len(_ckpt)
    _ckpt = _ckpt[~_ckpt["Method"].isin(MOFAPLUS_METHOD_NAMES)]
    _ckpt.to_csv(PILOT_CHECKPOINT_PATH, index=False)
    print(f"pilot_checkpoint.csv: removed {_before - len(_ckpt)} MOFA+ row(s), {len(_ckpt)} remain")

# 2) pilot_imputed_matrices/: delete the saved MOFA+ matrix files, so that Cells 8-10's
#    "checkpoint and saved matrix both present" check no longer finds them
_removed = 0
for f in PILOT_MATRICES_DIR.glob("pilot_*_imputed_MOFA+*.csv"):
    f.unlink()
    _removed += 1
print(f"pilot_imputed_matrices/: removed {_removed} MOFA+ matrix file(s)")

# 3) benchmark_checkpoint_<matrix>.csv: drop the MOFA+_native full-grid rows (Cell 11)
for matrix_name in MATRICES:
    ckpt_path = RESULTS_DIR / f"benchmark_checkpoint_{matrix_name}.csv"
    if not ckpt_path.exists():
        continue
    df = pd.read_csv(ckpt_path)
    before = len(df)
    df = df[df["Method"] != "MOFA+_native"]
    df.to_csv(ckpt_path, index=False)
    print(f"benchmark_checkpoint_{matrix_name}.csv: removed {before - len(df)} MOFA+_native row(s), {len(df)} remain")

pilot_checkpoint.csv: removed 9 MOFA+ row(s), 53 remain


pilot_imputed_matrices/: removed 9 MOFA+ matrix file(s)
benchmark_checkpoint_dlbcl_rna.csv: removed 5 MOFA+_native row(s), 200 remain
benchmark_checkpoint_dlbcl_protein.csv: removed 5 MOFA+_native row(s), 385 remain
benchmark_checkpoint_crc_protein.csv: removed 5 MOFA+_native row(s), 385 remain


In [9]:
# --- Cell 8: analysis 1, per-matrix baseline ("MOFA+_native") ------------
# Checkpointed into the same pilot_checkpoint.csv / pilot_imputed_matrices/ that the
# 25-method benchmark uses, so MOFA+'s rows land alongside theirs.
if PILOT_CHECKPOINT_PATH.exists():
    _checkpoint_df = pd.read_csv(PILOT_CHECKPOINT_PATH)
    pilot_rows = _checkpoint_df.to_dict("records")
else:
    pilot_rows = []
_pilot_row_index = {(r["Matrix"], r["Method"]): i for i, r in enumerate(pilot_rows)}


def _pilot_matrix_path(matrix_name, method_name):
    return PILOT_MATRICES_DIR / f"pilot_{matrix_name}_imputed_{method_name}.csv"


def _record_pilot_result(matrix_name, method_name, m, X_hat):
    matrix_path = _pilot_matrix_path(matrix_name, method_name)
    X_hat.rename_axis("Feature_ID").to_csv(matrix_path)
    if (matrix_name, method_name) in _pilot_row_index:
        pilot_rows[_pilot_row_index[(matrix_name, method_name)]] = m
    else:
        pilot_rows.append(m)
        _pilot_row_index[(matrix_name, method_name)] = len(pilot_rows) - 1
    pd.DataFrame(pilot_rows).to_csv(PILOT_CHECKPOINT_PATH, index=False)


if not MOFAPLUS_AVAILABLE:
    print("skipping analysis 1 (MOFA+ native, per-matrix) - mofapy2 not available")
else:
    for matrix_name, X in MATRICES.items():
        method_name = "MOFA+_native"
        matrix_path = _pilot_matrix_path(matrix_name, method_name)
        if (matrix_name, method_name) in _pilot_row_index and matrix_path.exists():
            print(f"{matrix_name}/{method_name}: skipped - checkpoint + saved matrix both present")
            continue
        print(f"\n=== MOFA+_native: {matrix_name} {X.shape} ===")
        Y_masked, mask, mechanism = build_masked_scenario(PILOT_ROW, X)
        t0 = time.time()
        try:
            recon, _ = run_mofaplus({matrix_name: Y_masked})
            m, X_hat = score_full(matrix_name, method_name, X, recon[matrix_name], mask, t0)
        except Exception as exc:
            m = {"Matrix": matrix_name, "Method": method_name, "Seconds": time.time() - t0,
                 "RMSE": np.nan, "Error": f"{type(exc).__name__}: {exc}"}
            X_hat = None
            print(f"  !! {method_name} failed on {matrix_name}: {exc}")
        if X_hat is not None:
            _record_pilot_result(matrix_name, method_name, m, X_hat)
        print(f"  RMSE={m.get('RMSE', float('nan')):.4f}  ({m['Seconds']:.2f}s)")


=== MOFA+_native: dlbcl_rna (500, 318) ===

        #########################################################
        ###           __  __  ____  ______                    ### 
        ###          |  \/  |/ __ \|  ____/\    _             ### 
        ###          | \  / | |  | | |__ /  \ _| |_           ### 
        ###          | |\/| | |  | |  __/ /\ \_   _|          ###
        ###          | |  | | |__| | | / ____ \|_|            ###
        ###          |_|  |_|\____/|_|/_/    \_\              ###
        ###                                                   ### 
        ######################################################### 
         




Successfully loaded view='dlbcl_rna' group='all_samples' with N=318 samples and D=500 features...


Model options:
- Automatic Relevance Determination prior on the factors: False
- Automatic Relevance Determination prior on the weights: True
- Spike-and-slab prior on the factors: False
- Spike-and-slab prior on the weights: True
Likelihoods:
- View 0 (dlbcl_rna): gaussian




######################################
## Training the model with seed 0 ##
######################################



Converged!



#######################
## Training finished ##
#######################


  RMSE=1.1714  (5.20s)

=== MOFA+_native: dlbcl_protein (500, 332) ===

        #########################################################
        ###           __  __  ____  ______                    ### 
        ###          |  \/  |/ __ \|  ____/\    _             ### 
        ###          | \  / | |  | | |__ /  \ _| |_           ### 
        ###          | |\/| | |  | |  __/ /\ \_   _|          ###
        ##

In [ ]:
# --- Cell 9: analysis 2, MNAR-aware pre-impute ablation (protein matrices only) ---
if not MOFAPLUS_AVAILABLE:
    print("skipping analysis 2 (pre-impute ablation) - mofapy2 not available")
else:
    PRE_IMPUTERS = {"QRILC": qrilc_impute, "MinProb": minprob_impute}
    for matrix_name in PROTEIN_MATRICES:
        X = MATRICES[matrix_name]
        Y_masked, mask, mechanism = build_masked_scenario(PILOT_ROW, X)
        for pre_name, pre_fn in PRE_IMPUTERS.items():
            method_name = f"MOFA+_after_{pre_name}"
            matrix_path = _pilot_matrix_path(matrix_name, method_name)
            if (matrix_name, method_name) in _pilot_row_index and matrix_path.exists():
                print(f"{matrix_name}/{method_name}: skipped - checkpoint + saved matrix both present")
                continue
            print(f"\n=== {method_name}: {matrix_name} {X.shape} ===")
            t0 = time.time()
            try:
                rng = np.random.default_rng(RNG_SEED)
                Y_pre_imputed = pre_fn(Y_masked, rng=rng)  # leaves no NaNs behind
                assert not Y_pre_imputed.isna().values.any(), f"{pre_name} left NaNs behind - check its fill logic"
                recon, _ = run_mofaplus({matrix_name: Y_pre_imputed})
                # Score against the mask (the artificially withheld positions), not
                # mofapy2's internal missing-mask, which is empty here --- see the Cell 7 docstring.
                m, X_hat = score_full(matrix_name, method_name, X, recon[matrix_name], mask, t0)
            except Exception as exc:
                m = {"Matrix": matrix_name, "Method": method_name, "Seconds": time.time() - t0,
                     "RMSE": np.nan, "Error": f"{type(exc).__name__}: {exc}"}
                X_hat = None
                print(f"  !! {method_name} failed on {matrix_name}: {exc}")
            if X_hat is not None:
                _record_pilot_result(matrix_name, method_name, m, X_hat)
            print(f"  RMSE={m.get('RMSE', float('nan')):.4f}  ({m['Seconds']:.2f}s)")


=== MOFA+_after_QRILC: dlbcl_protein (500, 332) ===



        #########################################################
        ###           __  __  ____  ______                    ### 
        ###          |  \/  |/ __ \|  ____/\    _             ### 
        ###          | \  / | |  | | |__ /  \ _| |_           ### 
        ###          | |\/| | |  | |  __/ /\ \_   _|          ###
        ###          | |  | | |__| | | / ____ \|_|            ###
        ###          |_|  |_|\____/|_|/_/    \_\              ###
        ###                                                   ### 
        ######################################################### 
         


Successfully loaded view='dlbcl_protein' group='all_samples' with N=332 samples and D=500 features...


Model options:
- Automatic Relevance Determination prior on the factors: False
- Automatic Relevance Determination prior on the weights: True
- Spike-and-slab prior on the factors: False
- Spike-and-slab prior on the weights: True
Likelihoods:
- View 0 (dlbcl_protein): gaussian




#

In [11]:
# --- Cell 10: analysis 3, joint two-view DLBCL RNA + protein ("MOFA+_joint") ---
# crc_protein has no joint partner (it's a separate cohort), so it stays native-only --
# already covered by analysis 1.
if not MOFAPLUS_AVAILABLE:
    print("skipping analysis 3 (joint two-view) - mofapy2 not available")
else:
    method_name = "MOFA+_joint"
    already_done = all(
        (mn, method_name) in _pilot_row_index and _pilot_matrix_path(mn, method_name).exists()
        for mn in JOINT_MATRICES
    )
    if already_done:
        print(f"joint run: skipped - checkpoint + saved matrices for {list(JOINT_MATRICES)} both present")
    else:
        print(f"\n=== MOFA+_joint: {[(n, d.shape) for n, d in JOINT_MATRICES.items()]} ===")
        masked_views, masks, truths = {}, {}, {}
        for matrix_name, X in JOINT_MATRICES.items():
            Y_masked, mask, mechanism = build_masked_scenario(PILOT_ROW, X)
            masked_views[matrix_name] = Y_masked
            masks[matrix_name] = mask
            truths[matrix_name] = X
        t0 = time.time()
        try:
            recon, _ = run_mofaplus(masked_views)
            for matrix_name in JOINT_MATRICES:
                m, X_hat = score_full(matrix_name, method_name, truths[matrix_name], recon[matrix_name],
                                       masks[matrix_name], t0)
                _record_pilot_result(matrix_name, method_name, m, X_hat)
                print(f"  {matrix_name}: RMSE={m.get('RMSE', float('nan')):.4f}")
        except Exception as exc:
            print(f"  !! {method_name} failed: {type(exc).__name__}: {exc}")


=== MOFA+_joint: [('dlbcl_rna', (500, 332)), ('dlbcl_protein', (500, 332))] ===

        #########################################################
        ###           __  __  ____  ______                    ### 
        ###          |  \/  |/ __ \|  ____/\    _             ### 
        ###          | \  / | |  | | |__ /  \ _| |_           ### 
        ###          | |\/| | |  | |  __/ /\ \_   _|          ###
        ###          | |  | | |__| | | / ____ \|_|            ###
        ###          |_|  |_|\____/|_|/_/    \_\              ###
        ###                                                   ### 
        ######################################################### 
         




Successfully loaded view='dlbcl_rna' group='all_samples' with N=332 samples and D=500 features...
Successfully loaded view='dlbcl_protein' group='all_samples' with N=332 samples and D=500 features...


Model options:
- Automatic Relevance Determination prior on the factors: False
- Automatic Relevance Determination prior on the weights: True
- Spike-and-slab prior on the factors: False
- Spike-and-slab prior on the weights: True
Likelihoods:
- View 0 (dlbcl_rna): gaussian
- View 1 (dlbcl_protein): gaussian




######################################
## Training the model with seed 0 ##
######################################



Converged!



#######################
## Training finished ##
#######################


  dlbcl_rna: RMSE=1.1904
  dlbcl_protein: RMSE=0.5286


In [ ]:
# --- Cell 11: analysis 1 extended, full-grid MOFA+_native --------------
# (SLOW-tier scenario subset, -- now optionally the FULL grid, see
# MOFAPLUS_FULL_GRID_ALL_REPLICATES below)

# MOFA+_native only - the variant directly comparable to the 25 conventional
# methods. Analyses 2 (pre-impute ablation) and 3 (joint) stay pilot-scenario-only
# (Cells 9-10); they're validity checks, not part of the full grid.

# The scenario subset used to be restricted to SLOW_NAMES-tier logic: Replicate == 1
# AND Masking_Fraction in SLOW_GRID_MASKING AND MNAR_Fraction in SLOW_GRID_MNAR - the
# same reduction benchmark_imputation_clinical.ipynb/py applies to RandomForest,
# MissForest, DAE, VAE, DreamAI, ADMIN, and EM_MVN, since MOFA+'s training cost falls
# in that same tier. With the current grid (MASKING_FRACTIONS=[0.30], which is also
# SLOW_GRID_MASKING's only value, and MNAR_FRACTIONS == SLOW_GRID_MNAR exactly), the
# masking/MNAR part of that filter no longer excludes anything on its own - the only
# thing still capping N below the full grid is Replicate == 1.
# MOFAPLUS_FULL_GRID_ALL_REPLICATES below mirrors benchmark_imputation_clinical.ipynb/py's
# FULL_GRID_FOR_ALL_METHODS toggle: True runs every replicate too, so MOFA+_native's N
# matches the conventional methods' full grid (25 with the current grid) instead of being
# capped at N_REPLICATES=1's worth of scenarios (5). MASKING_FRACTIONS, MNAR_FRACTIONS,
# N_REPLICATES, and the Seed formula are all the same, so Scenario_ID and Seed
# line up exactly with benchmark_imputation_clinical.ipynb/py's own experiment_design table
# - necessary for the two files' rows to merge cleanly in
# evaluate_imputation_methods_multiomic_clinical.ipynb's Cell 8/9.

# Checkpointed into the same benchmark_checkpoint_<matrix>.csv files that
# benchmark_imputation_clinical.ipynb/py's Cell 16 writes (not pilot_checkpoint.csv), so
# MOFA+_native rows land alongside the 25 methods' full-grid rows and appear
# automatically in Fig 1-4 and benchmark_results_aggregated.csv - no separate
# table to cross-reference. Don't run this cell at the same time as
# benchmark_imputation_clinical.ipynb/py's Cell 16 for the same matrix: both do a
# read-modify-write cycle against the same checkpoint file, and running them
# concurrently risks one process's write clobbering the other's in-flight update.
# Run them sequentially instead - either order works, since both resume from
# whatever is already checkpointed.
#

MASKING_FRACTIONS = [0.30]  
                                   
MNAR_FRACTIONS = [0.00, 0.25, 0.50, 0.75, 1.00]
N_REPLICATES = 5  # -> 50 scenarios total, same grid as benchmark_imputation_clinical.ipynb/py / 
                  # evaluate_imputation_methods_clinical.ipynb

SLOW_GRID_MASKING = {0.30}  # matches benchmark_imputation_clinical.ipynb/py's SLOW_NAMES tier
SLOW_GRID_MNAR = {0.00, 0.25, 0.50, 0.75, 1.00}

# Mirrors benchmark_imputation_clinical.ipynb/py's FULL_GRID_FOR_ALL_METHODS: True runs
# MOFA+_native on every replicate, not just Replicate == 1. Big runtime jump - MOFA+
# training is the single most expensive method in this whole benchmark, so this multiplies
# its full-grid training-run count by N_REPLICATES (5x with the current grid) across all
# three matrices. Time one matrix before committing to all three. Set back to False to
# return to the Replicate == 1 -only subset.
MOFAPLUS_FULL_GRID_ALL_REPLICATES = True

_rows = []
_scenario_id = 0
for mf in MASKING_FRACTIONS:
    for nf in MNAR_FRACTIONS:
        for rep in range(1, N_REPLICATES + 1):
            _scenario_id += 1
            _rows.append({"Scenario_ID": _scenario_id, "Masking_Fraction": mf, "MNAR_Fraction": nf,
                          "Replicate": rep, "Seed": RNG_SEED * 10_000 + _scenario_id})
experiment_design = pd.DataFrame(_rows)

_replicate_ok = (
    experiment_design["Replicate"] >= 1  # no-op: every replicate qualifies
    if MOFAPLUS_FULL_GRID_ALL_REPLICATES
    else experiment_design["Replicate"] == 1
)
mofaplus_full_grid_design = experiment_design[
    _replicate_ok
    & (experiment_design["Masking_Fraction"].isin(SLOW_GRID_MASKING))
    & (experiment_design["MNAR_Fraction"].isin(SLOW_GRID_MNAR))
].reset_index(drop=True)
print(
    f"MOFA+_native full-grid scenario subset: {len(mofaplus_full_grid_design)} of {len(experiment_design)} "
    f"scenarios x {len(MATRICES)} matrices = {len(mofaplus_full_grid_design) * len(MATRICES)} total training runs"
)

FULL_GRID_METHOD_NAME = "MOFA+_native"


def run_mofaplus_full_grid(matrix_name, X, checkpoint_path):
    """Follows the same skip/resume/checkpoint-after-every-scenario contract as
    benchmark_imputation_clinical.ipynb/py's run_full_benchmark(), but restricted to
    mofaplus_full_grid_design and a single method."""
    if checkpoint_path.exists():
        all_results = pd.read_csv(checkpoint_path).to_dict("records")
        print(f"{matrix_name}: resuming from checkpoint, {len(all_results)} result(s) already in file.")
    else:
        all_results = []
    already_done = {(r["Scenario_ID"], r["Method"]) for r in all_results}

    for _, row in mofaplus_full_grid_design.iterrows():
        if (row["Scenario_ID"], FULL_GRID_METHOD_NAME) in already_done:
            continue
        Y_masked, mask, mechanism = build_masked_scenario(row, X)
        mcar_only = pd.DataFrame(mechanism.values == "MCAR", index=mask.index, columns=mask.columns)
        mnar_only = pd.DataFrame(mechanism.values == "MNAR", index=mask.index, columns=mask.columns)
        t0 = time.time()
        try:
            recon, _ = run_mofaplus({matrix_name: Y_masked}, seed=int(row["Seed"]))
            X_hat = recon[matrix_name].reindex(index=X.index, columns=X.columns)
            overall = calculate_metrics(X, X_hat, mask)
            rec = {"Scenario_ID": row["Scenario_ID"], "Method": FULL_GRID_METHOD_NAME,
                   "Masking_Fraction": row["Masking_Fraction"], "MNAR_Fraction": row["MNAR_Fraction"],
                   "Replicate": row["Replicate"], "Seconds": time.time() - t0, **overall}
            if mcar_only.values.any():
                rec["RMSE_MCAR_subset"] = calculate_metrics(X, X_hat, mcar_only)["RMSE"]
            if mnar_only.values.any():
                rec["RMSE_MNAR_subset"] = calculate_metrics(X, X_hat, mnar_only)["RMSE"]
        except Exception as exc:
            rec = {"Scenario_ID": row["Scenario_ID"], "Method": FULL_GRID_METHOD_NAME,
                   "Masking_Fraction": row["Masking_Fraction"], "MNAR_Fraction": row["MNAR_Fraction"],
                   "Replicate": row["Replicate"], "Seconds": time.time() - t0, "RMSE": np.nan,
                   "Error": f"{type(exc).__name__}: {exc}"}
            print(f"    !! {matrix_name}/{FULL_GRID_METHOD_NAME} failed on scenario {row['Scenario_ID']}: {exc}")
        all_results.append(rec)
        already_done.add((row["Scenario_ID"], FULL_GRID_METHOD_NAME))
        pd.DataFrame(all_results).to_csv(checkpoint_path, index=False)  # checkpoint after each scenario
        print(
            f"  scenario {int(row['Scenario_ID']):3d} (mask={row['Masking_Fraction']:.2f}, "
            f"mnar={row['MNAR_Fraction']:.2f}): RMSE={rec.get('RMSE', float('nan')):.4f} ({rec['Seconds']:.1f}s)"
        )

    return pd.DataFrame(all_results)


# Run one matrix at a time (uncomment to run) - each call resumes from its own
# checkpoint if interrupted, so it's safe to run matrices in separate sessions.
# Time the first matrix before committing to all three: the "fast" convergence_mode
# and 1000-iter cap (Cell 7) were tuned from the pilot run only, and haven't yet
# been timed at this scale.
mofaplus_full_grid_results = {}
if MOFAPLUS_AVAILABLE:
     for matrix_name, X in MATRICES.items():
         print(f"\n=== full-grid MOFA+_native: {matrix_name} {X.shape} ===")
         mofaplus_full_grid_results[matrix_name] = run_mofaplus_full_grid(
             matrix_name, X, RESULTS_DIR / f"benchmark_checkpoint_{matrix_name}.csv"
         )

MOFA+_native full-grid scenario subset: 25 of 25 scenarios x 3 matrices = 75 total training runs

=== full-grid MOFA+_native: dlbcl_rna (500, 318) ===
dlbcl_rna: resuming from checkpoint, 200 result(s) already in file.

        #########################################################
        ###           __  __  ____  ______                    ### 
        ###          |  \/  |/ __ \|  ____/\    _             ### 
        ###          | \  / | |  | | |__ /  \ _| |_           ### 
        ###          | |\/| | |  | |  __/ /\ \_   _|          ###
        ###          | |  | | |__| | | / ____ \|_|            ###
        ###          |_|  |_|\____/|_|/_/    \_\              ###
        ###                                                   ### 
        ######################################################### 
         




Successfully loaded view='dlbcl_rna' group='all_samples' with N=318 samples and D=500 features...


Model options:
- Automatic Relevance Determination prior on the factors: False
- Automatic Relevance Determination prior on the weights: True
- Spike-and-slab prior on the factors: False
- Spike-and-slab prior on the weights: True
Likelihoods:
- View 0 (dlbcl_rna): gaussian




######################################
## Training the model with seed 1 ##
######################################



Converged!



#######################
## Training finished ##
#######################


  scenario   1 (mask=0.30, mnar=0.00): RMSE=1.1756 (5.5s)

        #########################################################
        ###           __  __  ____  ______                    ### 
        ###          |  \/  |/ __ \|  ____/\    _             ### 
        ###          | \  / | |  | | |__ /  \ _| |_           ### 
        ###          | |\/| | |  | |  __/ /\ \_   _|          ###
        ###          | 

In [13]:
# --- Cell 12: combined comparison table -------------------------------------
# Reloads from PILOT_CHECKPOINT_PATH rather than the in-memory pilot_rows, so
# this cell reflects whatever combination of the 25 conventional methods and
# MOFA+ runs has actually completed and been checkpointed so far.
pilot_checkpoint = pd.read_csv(PILOT_CHECKPOINT_PATH) if PILOT_CHECKPOINT_PATH.exists() else pd.DataFrame()
mofaplus_methods = {"MOFA+_native", "MOFA+_after_QRILC", "MOFA+_after_MinProb", "MOFA+_joint"}
comparison_cols = [
    "Matrix", "Method", "MAE", "RMSE", "NRMSE",
    "Pearson_r_raw_pooled", "Spearman_r_raw_pooled",
    "Pearson_r_feature_centered_pooled", "Spearman_r_feature_centered_pooled",
    "N_Evaluated", "N_Features_Pooled", "Seconds",
]
if not pilot_checkpoint.empty:
    mofaplus_summary = pilot_checkpoint[pilot_checkpoint["Method"].isin(mofaplus_methods)]
    mofaplus_summary = mofaplus_summary[[c for c in comparison_cols if c in mofaplus_summary.columns]]
    mofaplus_summary = mofaplus_summary.sort_values(["Matrix", "Method"])
    mofaplus_summary.to_csv(RESULTS_DIR / "mofaplus_pilot_summary.csv", index=False)
    print(f"MOFA+ rows in the shared pilot checkpoint: {len(mofaplus_summary)} of {len(pilot_checkpoint)} total")
else:
    mofaplus_summary = pd.DataFrame(columns=comparison_cols)
    print("pilot_checkpoint.csv not found or empty -- nothing to summarize yet")
mofaplus_summary

MOFA+ rows in the shared pilot checkpoint: 9 of 60 total


,Matrix,Method,MAE,RMSE,NRMSE,Pearson_r_raw_pooled,Spearman_r_raw_pooled,Pearson_r_feature_centered_pooled,Spearman_r_feature_centered_pooled,N_Evaluated,N_Features_Pooled,Seconds
55,crc_protein,MOFA+_after_MinProb,1.343597,2.030204,0.183096,0.819939,0.866166,0.246611,0.395856,47848,500.0,5.991402
54,crc_protein,MOFA+_after_QRILC,1.736806,2.510587,0.161821,0.843096,0.885638,0.331643,0.426728,47848,500.0,6.057828
53,crc_protein,MOFA+_native,0.862236,1.295767,0.105017,0.911148,0.905854,0.529146,0.543607,47848,500.0,10.916647
57,dlbcl_protein,MOFA+_after_MinProb,0.890983,1.158108,0.291612,0.921344,0.863074,0.081028,0.081925,6615,319.0,3.290473
56,dlbcl_protein,MOFA+_after_QRILC,0.628953,0.890098,0.216149,0.945134,0.901211,0.276704,0.230814,6615,319.0,16.522891
59,dlbcl_protein,MOFA+_joint,0.367918,0.528593,0.123403,0.968028,0.939430,0.574137,0.528656,6615,319.0,6.653965
52,dlbcl_protein,MOFA+_native,0.330276,0.486366,0.112191,0.973215,0.946125,0.661573,0.621459,6615,319.0,3.028347
58,dlbcl_rna,MOFA+_joint,0.888788,1.190371,0.160354,0.800429,0.789772,0.677574,0.632409,15900,500.0,6.382025
51,dlbcl_rna,MOFA+_native,0.870228,1.171405,0.157658,0.807521,0.796776,0.690482,0.647995,15900,500.0,3.136245
